# Online Retail Analysis Initial Data Review

In [3]:
import pandas as pd

df = pd.read_excel('data/raw/Online Retail.xlsx')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.shape

(541909, 8)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [6]:
missing_values = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100)
})

missing_values.sort_values(by="missing_count", ascending=False)

,missing_count,missing_percent
CustomerID,135080,24.926694
Description,1454,0.268311
StockCode,0,0.000000
InvoiceNo,0,0.000000
Quantity,0,0.000000
InvoiceDate,0,0.000000
UnitPrice,0,0.000000
Country,0,0.000000


In [7]:
df[["Quantity", "UnitPrice"]].describe()

,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


In [8]:
df["InvoiceDate"].agg(['min', 'max'])

min   2010-12-01 08:26:00
max   2011-12-09 12:50:00
Name: InvoiceDate, dtype: datetime64[us]

In [9]:
df.duplicated().sum()

np.int64(5268)

In [10]:
cancellation_mask = (
    df["InvoiceNo"]
    .astype("string")
    .str.upper()
    .str.startswith("C", na=False)
)

df.loc[
    cancellation_mask,
    ["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "InvoiceDate"]
].head(10)

,InvoiceNo,StockCode,Quantity,UnitPrice,InvoiceDate
141,C536379,D,-1,27.50,2010-12-01 09:41:00
154,C536383,35004C,-1,4.65,2010-12-01 09:49:00
235,C536391,22556,-12,1.65,2010-12-01 10:24:00
236,C536391,21984,-24,0.29,2010-12-01 10:24:00
237,C536391,21983,-24,0.29,2010-12-01 10:24:00
238,C536391,21980,-24,0.29,2010-12-01 10:24:00
239,C536391,21484,-12,3.45,2010-12-01 10:24:00
240,C536391,22557,-12,1.65,2010-12-01 10:24:00
241,C536391,22553,-24,1.65,2010-12-01 10:24:00
939,C536506,22960,-6,4.25,2010-12-01 12:38:00


## Initial observations

For readability, I display decimal values to a maximum of two decimal places in this summary. Additional decimal places would not meaningfully improve these initial observations. This is a presentation choice; the original values remain unchanged for calculations.

### What is in the dataset?

The dataset contains **541,909 rows**. This is not necessarily the number of orders, since one order can include several products.

Most columns loaded into formats that fit their contents: quantities are whole numbers, prices allow decimal values, and invoice dates include both dates and times.

A few columns need a closer look:

- `InvoiceNo`, `StockCode`, and `Description` loaded as `object`, which can hold text or a mixture of value types. This is not necessarily a problem.
- `CustomerID` loaded as a decimal number, with values such as `17850.0`. This likely happened because some customer IDs are missing. These values identify customers; they are not numbers we should average or add together.

### Missing information

About **24.93% of rows are missing a customer ID**, and **0.27% are missing a product description**.

Missing customer IDs will make it harder to study individual customers and repeat purchases. However, those rows may still be useful for studying sales, so I will not remove them automatically.

One possibility is to recover a missing customer ID from another line on the same invoice. I will check records with the same invoice number and exact date and time:

- If the group has one known customer ID, it may be possible to fill the gap.
- If every customer ID is missing, this approach will not help.
- If the group contains different customer IDs, I will investigate before making changes.

I will also check whether invoice numbers appear with different dates or customer IDs. Any filled values will be recorded separately so the original data remains available.

For missing product descriptions, I will check whether the same product code has a description elsewhere in the dataset.

### Unusual quantities and prices

Most quantities and prices are much smaller than the largest values in the dataset:

| Measure | Quantity | Unit price (£) |
|---|---:|---:|
| Lowest value | -80,995 | -11,062.06 |
| Middle value (median) | 3 | 2.08 |
| Average | 9.55 | 4.61 |
| Highest value | 80,995 | 38,970 |

The middle half of quantities falls between **1 and 10**, while the middle half of unit prices falls between **£1.25 and £4.13**. The extreme values deserve attention because they could strongly affect the results.

Negative quantities can represent cancellations. I will check whether that explains all negative quantities.

The quantities of **80,995 and -80,995** could be an order and its cancellation. However, matching amounts alone do not prove that they belong together or that the original order was a mistake. I will compare the details of those records.

Negative prices also need investigation. I will not assume they have the same explanation as negative quantities.

### Time period covered

The data runs from **December 1, 2010 at 8:26 a.m.** to **December 9, 2011 at 12:50 p.m.**

December 2011 only covers part of the month. Comparing its total sales directly with a full month could create a misleading impression of a decline.

### Repeated rows

The initial check found **5,268 rows that exactly repeat an earlier row**.

These may be duplicate records, but I will examine them before deleting anything. I need to understand whether they are recording mistakes or legitimate repeated entries.

### What I will check next

1. Whether missing customer IDs can be recovered from other lines on the same invoice.
2. Whether missing descriptions can be found using product codes.
3. What explains negative quantities, negative prices, and unusually large values.
4. Whether repeated rows should be removed.
5. Whether the date coverage supports fair monthly comparisons.

I will keep the original data unchanged and document any changes made during cleaning.